In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoAlertPresentException
from datetime import datetime
import time
import traceback

TARGET_URL     = "https://isrc.snu.ac.kr/hm/route/progress/schedule/view?sorderSkey=13277"
PROCESS_NAME   = "ICP Metal Etcher (03)"
START_DATETIME = "2026-07-04 15:00"
END_DATETIME   = "2026-07-04 16:00"

def dismiss_alert(driver):
    try:
        alert = driver.switch_to.alert
        print(f"  [알림 닫음: '{alert.text}']")
        alert.accept()
        time.sleep(0.3)
        return True
    except NoAlertPresentException:
        return False

def js_set(driver, element, value):
    driver.execute_script("""
        var el = arguments[0], val = arguments[1];
        el.value = val;
        el.dispatchEvent(new Event('input',  {bubbles:true}));
        el.dispatchEvent(new Event('change', {bubbles:true}));
    """, element, value)

def find_예약_btn(driver):
    candidates = [
        el for el in driver.find_elements(By.XPATH,
            "//button[contains(.,'예약') and not(contains(.,'신청'))]"
            " | //a[contains(.,'예약') and not(contains(.,'신청'))]")
        if el.is_displayed()
    ]
    return candidates[-1] if candidates else None

def click_ok_dialog(driver, timeout=8):
    """'예약하시겠습니까?' 다이얼로그의 Ok 버튼만 정확히 클릭"""
    for i in range(timeout * 5):
        time.sleep(0.2)
        try:
            driver.switch_to.default_content()
        except Exception:
            pass

        confirm_els = [el for el in driver.find_elements(By.XPATH,
            "//*[contains(text(),'예약하시겠습니까')]")
            if el.is_displayed()]
        if not confirm_els:
            continue

        print(f"  '예약하시겠습니까?' 감지 (시도#{i+1})")

        ok_btn = driver.execute_script("""
            var textNodes = [];
            var walker = document.createTreeWalker(
                document.body, NodeFilter.SHOW_TEXT, null, false);
            var node;
            while (node = walker.nextNode()) {
                if (node.textContent.includes('예약하시겠습니까')) {
                    textNodes.push(node.parentElement);
                }
            }
            for (var el of textNodes) {
                var p = el;
                for (var i = 0; i < 10; i++) {
                    if (!p || p.tagName === 'BODY') break;
                    var btns = p.querySelectorAll('button, a');
                    for (var btn of btns) {
                        var t = btn.textContent.trim();
                        if (t === 'Ok' || t === 'OK') return btn;
                    }
                    for (var btn of btns) {
                        var t = btn.textContent.trim();
                        if (t === '확인') return btn;
                    }
                    p = p.parentElement;
                }
            }
            return null;
        """)

        if ok_btn:
            tag = ok_btn.tag_name
            txt = driver.execute_script("return arguments[0].textContent.trim();", ok_btn)
            print(f"  Ok버튼 (다이얼로그 내): <{tag}> '{txt}'")

            try:
                ok_btn.click()
                print("  → native click")
            except Exception:
                pass
            time.sleep(0.15)

            try:
                ActionChains(driver).move_to_element(ok_btn).click().perform()
                print("  → ActionChains")
            except Exception:
                pass
            time.sleep(0.15)

            try:
                ok_btn.send_keys(Keys.RETURN)
                print("  → send_keys ENTER")
            except Exception:
                pass
            time.sleep(0.15)

            try:
                driver.switch_to.active_element.send_keys(Keys.RETURN)
                print("  → active_element ENTER")
            except Exception:
                pass

            time.sleep(0.5)
            return True

        print(f"  ⚠️ 다이얼로그 내 Ok버튼 못 찾음. 전체 visible:")
        for tag in ['button', 'a']:
            visible = [(e.text.strip(), e.is_displayed()) for e in
                       driver.find_elements(By.TAG_NAME, tag) if e.text.strip()]
            if visible:
                print(f"    <{tag}>: {visible[:10]}")

    print("  ⚠️ '예약하시겠습니까?' 텍스트 미감지")
    return dismiss_alert(driver)

def run_booking():
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 15)

    try:
        print("STEP 1: 페이지 이동...")
        driver.get(TARGET_URL)
        time.sleep(1)
        print(f"  URL: {driver.current_url}")
        if "login" in driver.current_url.lower():
            print("  ⚠️ 로그인 필요 → 브라우저에서 로그인하고 mypage → 진행할 공정의뢰 목록에서 공정 스케줄 관리 클릭 후 Enter")
            input("  공정 스케줄 관리 클릭 완료 후 Enter: ")
            driver.get(TARGET_URL)
            time.sleep(1)

        print("STEP 2-5: 로드 → 드롭다운 → 입력 → 검색...")
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "table")))
        time.sleep(1)
        Select(driver.find_element(By.NAME, "searchType")).select_by_value("route")
        time.sleep(0.3)
        inp = driver.find_element(By.NAME, "typeValue")
        inp.clear()
        inp.send_keys(PROCESS_NAME)
        driver.find_element(By.XPATH, "//a[contains(text(),'검색')]").click()
        time.sleep(1.5)

        print("STEP 5-2: 공정스케줄 상태 → 대기만 체크 후 재검색...")
        # all과 STB는 건드리지 않고 나머지 6개만 개별 해제
        # → all 해제 시 JS 연동으로 STB까지 풀리는 문제를 원천 차단
        for val in ["PLN", "CNL", "DCL", "PRC", "RSL", "CFM"]:
            cb = driver.find_element(By.XPATH, f"//input[@name='schedStatus' and @value='{val}']")
            if cb.is_selected():
                driver.execute_script("arguments[0].click();", cb)
            time.sleep(0.1)
        time.sleep(0.3)
        driver.find_element(By.XPATH, "//a[contains(@class,'btn_search')]").click()
        time.sleep(2)

        print("STEP 6: 장비예약 클릭...")
        a_list = [
            el for el in driver.find_elements(By.XPATH, "//a[normalize-space(.)='장비예약']")
            if not (el.get_attribute("href") or "").startswith("http")
        ]
        btn_list = driver.find_elements(By.XPATH, "//button[normalize-space(.)='장비예약']")
        target = (btn_list or a_list)[0]
        driver.execute_script("arguments[0].scrollIntoView(true);", target)
        time.sleep(0.3)
        driver.execute_script("arguments[0].click();", target)
        time.sleep(1)

        print("STEP 7: 날짜/시간 직접 입력...")
        fmt = "%Y-%m-%d %H:%M"
        use_minutes = int((datetime.strptime(END_DATETIME, fmt)
                         - datetime.strptime(START_DATETIME, fmt)).total_seconds() / 60)
        js_set(driver, driver.find_element(By.NAME, "START_DT_STR"), START_DATETIME)
        print(f"  START_DT_STR = {START_DATETIME}")
        time.sleep(0.2); dismiss_alert(driver)
        js_set(driver, driver.find_element(By.NAME, "END_DT_STR"), END_DATETIME)
        print(f"  END_DT_STR   = {END_DATETIME}")
        time.sleep(0.2); dismiss_alert(driver)
        js_set(driver, driver.find_element(By.NAME, "USE_TIME"), str(use_minutes))
        print(f"  USE_TIME     = {use_minutes}분")
        time.sleep(0.3); dismiss_alert(driver)

        print("STEP 8: 예약신청 클릭...")
        original_window = driver.current_window_handle
        btn = driver.find_element(By.XPATH,
            "//a[contains(.,'예약신청')] | //button[contains(.,'예약신청')]")
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.5); dismiss_alert(driver)

        print("STEP 9: 공정 예약 팝업 → 예약 버튼...")
        예약_btn = None
        if len(driver.window_handles) > 1:
            for wh in driver.window_handles:
                if wh != original_window:
                    driver.switch_to.window(wh)
                    print("  새 창 전환됨"); break
            예약_btn = find_예약_btn(driver)

        if 예약_btn is None:
            driver.switch_to.default_content()
            iframes = driver.find_elements(By.TAG_NAME, "iframe")
            for idx, iframe in enumerate(iframes):
                try:
                    driver.switch_to.default_content()
                    driver.switch_to.frame(iframe)
                    b = find_예약_btn(driver)
                    if b:
                        예약_btn = b
                        print(f"  iframe[{idx}]에서 발견")
                        break
                except Exception:
                    pass

        if 예약_btn is None:
            driver.switch_to.default_content()
            예약_btn = find_예약_btn(driver)

        if 예약_btn is None:
            raise Exception("예약 버튼을 찾지 못함")

        try:
            예약_btn.click()
        except Exception:
            ActionChains(driver).move_to_element(예약_btn).click().perform()
        print("  예약 클릭")

        try:
            driver.switch_to.default_content()
        except Exception:
            pass
        time.sleep(1)

        print("STEP 10: 예약하시겠습니까? → Ok...")
        click_ok_dialog(driver, timeout=8)
        print("✅ 최종 예약 완료!")

    except Exception as e:
        print(f"\n❌ 오류: {type(e).__name__}: {e}")
        traceback.print_exc()
        dismiss_alert(driver)
    finally:
        input("\nEnter로 브라우저 닫기...")
        driver.quit()


run_booking()

STEP 1: 페이지 이동...
  URL: https://isrc.snu.ac.kr/login
  ⚠️ 로그인 필요 → 브라우저에서 로그인하고 mypage → 진행할 공정의뢰 목록에서 공정 스케줄 관리 클릭 후 Enter
STEP 2-5: 로드 → 드롭다운 → 입력 → 검색...
STEP 5-2: 공정스케줄 상태 → 대기만 체크 후 재검색...
STEP 6: 장비예약 클릭...
STEP 7: 날짜/시간 직접 입력...
  START_DT_STR = 2026-07-04 15:00
  END_DT_STR   = 2026-07-04 16:00
  USE_TIME     = 60분
STEP 8: 예약신청 클릭...
STEP 9: 공정 예약 팝업 → 예약 버튼...
  예약 클릭
STEP 10: 예약하시겠습니까? → Ok...


KeyboardInterrupt: 